In [ ]:
# Authentication
import os
from dotenv import load_dotenv
load_dotenv()
my_secret_key = os.getenv("notebook1_hf_key")
from huggingface_hub import login
login(token=my_secret_key)

In [ ]:
!pip install -q \
transformers==4.44.2 \
accelerate==0.33.0 \
bitsandbytes==0.43.3 \
sentencepiece \
fastapi \
uvicorn \
pyngrok \
huggingface_hub

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import os

MODEL_PATH = "./saved_model"
TOKENIZER_PATH = "./saved_tokenizer"

def load_model_and_tokenizer():
    if os.path.exists(MODEL_PATH) and os.path.exists(TOKENIZER_PATH):
        print("Loading saved model...")
        tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
        model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map="auto")
    else:
        print("Downloading model...")
        
        model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        tokenizer = AutoTokenizer.from_pretrained(model_id)
        tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )
        tokenizer.save_pretrained(TOKENIZER_PATH)
        model.save_pretrained(MODEL_PATH)

    return tokenizer, model


tokenizer, model = load_model_and_tokenizer()


def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import requests
from pyngrok import ngrok

app = FastAPI()

class PromptRequest(BaseModel):
    prompt: str

@app.post("/generate/")
async def generate(request: PromptRequest):
    # Summarize with Llama
    generated_text = generate_text(request.prompt)
    
    # Send to Notebook 2 for code processing
    notebook2_url = "https://trochoidally-ineffable-danna.ngrok-free.dev/generate/"
    code_response = requests.post(notebook2_url, json={"prompt": generated_text})
    
    return {
        "summary": generated_text,
        "code_response": code_response.json()
    }

# Start ngrok tunnel
ngrok_key = os.getenv("notebook1_ngrok_key")
ngrok.set_auth_token(ngrok_key)
public_url = ngrok.connect(8000)
print("Public URL:", public_url)

# Patch event loop and run uvicorn properly
nest_asyncio.apply()

config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, reload=True)
server = uvicorn.Server(config)
await server.serve()


In [ ]:
# curl.exe -X POST "https://ab73-34-168-13-97.ngrok-free.app/generate/" -H "Content-Type: application/json" -d "{\"prompt\":\"Give Python code to solve the 0/1 Knapsack Problem using dynamic programming.\"}" | jq